In [1]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath('..'))

In [2]:
# Import the functions you've built across the project
from quantum_qr.generator import generate
from quantum_qr.verifier import verify

# 1. Generate a known-authentic QR code
qr_path = "data/auth_demo.png"
print(f"Generating authentic QR code at {qr_path}...")
generate("pay alice $10", qr_path)

# 2. Run the quantum verifier pipeline
print("Running quantum verification...")
result = verify(qr_path)

# 3. Display the results clearly
print("\n--- Verification Results ---")
print(f"Verdict:          {result['verdict']}")
print(f"Measured Secret:  {result['measured_secret']}")
print(f"Classical Secret: {result['classical_secret']}")
print(f"Sanity Check:     {result['agree']}")
print(f"Raw Counts:       {result['counts']}")

# 4. Programmatic Assertions (Confirming the exact task requirements)
assert result["verdict"] == "authentic", f"Test failed! Expected 'authentic', got '{result['verdict']}'"
assert all(b == '0' for b in result["measured_secret"]), "Test failed! Measured secret is not all zeros."
assert result["agree"] is True, "Test failed! The quantum measurement did not match the classical secret."

print("\n✅ Success! The pipeline perfectly verified an authentic QR code.")

Generating authentic QR code at data/auth_demo.png...
Running quantum verification...

--- Verification Results ---
Verdict:          authentic
Measured Secret:  00000000
Classical Secret: 00000000
Sanity Check:     True
Raw Counts:       {'00000000': 1024}

✅ Success! The pipeline perfectly verified an authentic QR code.


In [5]:
import json
from quantum_qr.verifier import verify

# 1. Load the manifest
manifest_path = "../data/fixtures/manifest.json"
with open(manifest_path, 'r') as f:
    manifest = json.load(f)

# 2. Extract the expected data for our chosen fixture
fixture_name = "fixture_01_data.png"
fixture_path = f"../data/fixtures/{fixture_name}"

# FIX: Loop through the list to find our specific fixture
expected_secret = None
for item in manifest:
    # Note: If your JSON uses "file" or "name" instead of "filename", change it here!
    if item.get("filename") == fixture_name or item.get("file") == fixture_name:
        expected_secret = item["expected_secret"]
        break

if expected_secret is None:
    raise ValueError(f"Could not find {fixture_name} in the manifest! Check the JSON structure.")

# 3. Run the verifier pipeline
print(f"Verifying tampered fixture: {fixture_path}...")
result = verify(fixture_path)

# 4. Display the results
print("\n--- Tamper Verification Results ---")
print(f"Verdict:          {result['verdict']}")
print(f"Measured Secret:  {result['measured_secret']}")
print(f"Expected Secret:  {expected_secret}")

# 5. Programmatic Assertions (The real test!)
assert result["verdict"] == "tampered", f"Failed! Expected 'tampered', got '{result['verdict']}'"

# If this next assertion fails, it means our [::-1] reversal in verifier.py isn't aligned
# with how the manifest was generated. 
assert result["measured_secret"] == expected_secret, (
    f"Endianness Mismatch! \n"
    f"Measured: {result['measured_secret']}\n"
    f"Expected: {expected_secret}"
)

print("\n✅ Success! The tampered QR was caught, and the bit-ordering matches the manifest exactly.")

Verifying tampered fixture: ../data/fixtures/fixture_01_data.png...

--- Tamper Verification Results ---
Verdict:          tampered
Measured Secret:  00001010
Expected Secret:  00001010

✅ Success! The tampered QR was caught, and the bit-ordering matches the manifest exactly.


### Design Note: Qiskit Bit Ordering and Endianness

**The Problem:**
When constructing the oracle, our classical secret string is read left-to-right (index `0` to `n-1`), mapping bit `i` to qubit `i`. However, Qiskit represents measured states using a little-endian convention, placing qubit `0` ($q_0$) at the far right of the output bitstring ($q_{n-1} \dots q_0$). Without intervention, the measured secret will appear reversed compared to the classical classical secret.

**The Solution:**
To maintain consistency throughout the pipeline and match standard classical cryptographic representations, we chose to **reverse the measured bitstring** at the end of the quantum execution. 

Inside `verifier.py`, the raw output from Qiskit's `get_counts()` is sliced via `raw_measured[::-1]`. This re-aligns the quantum output to standard big-endian (left-to-right) indexing, ensuring that `measured_secret` directly equates to the classical target without requiring downstream consumers to understand Qiskit internals.